In [10]:
import pandas as pd, numpy as np
from sklearn.neural_network import MLPRegressor as MLPRergessor
from sklearn.preprocessing import MinMaxScaler
import random

FILE = '/content/NN3_REDUCE_DATASET_CLEAN.xlsx'

df = pd.read_excel(FILE, header = None)

print("Shape:", df.shape)
print(df.head())

series_list = []
num_rows = df.shape[0]
print("So cot: ", num_rows)


for r in range(num_rows):
  row = df.iloc[r, :]      #Lay cot thu c
  row = pd.to_numeric(row, errors = 'coerce')  #ep so, loi -> NaN
  row = row.dropna()
  row = row.values.astype(float)
  series_list.append(row)


print("So chuoi lay duoc: ", len(series_list))
for i in range(len(series_list)):
  print(f"Series {i+1} length = {len(series_list[i])}")

  print(series_list[i])


  h, r, k = 18, 79, 10


Shape: (11, 126)
    0     1     2     3     4     5     6      7      8      9    ...    116  \
0  5093  5110  5029  5216  4899  5274  5315   5019   5269   5054  ...   4606   
1  4290  4976  5342  6175  6853  7405  8031   8558   8878   9061  ...   7534   
2  4876  4304  2828  3972  4104  4324  7828  19352  35900  49948  ...  42540   
3  5305  5147  5620  5367  5068  5652  5106   5872   5815   5664  ...   4327   
4  4608  4704  4820  4855  4719  4482  4316   4554   4528   4396  ...   5042   

     117    118   119   120   121   122     123     124     125  
0   4507   4600  4594  4499  4602  4814  4824.0  4480.0  4998.0  
1   8040   8402  1895  2316  2866  3378  3970.0  4510.0  5033.0  
2  53804  55024  1000  1532  1968  2400  3224.0  4264.0  5304.0  
3   4401   4556  3413  3451  3941  4706  4011.0  4927.0  4585.0  
4   4796   4963  4848  5026  5105  4984  4976.0  5165.0  5007.0  

[5 rows x 126 columns]
So cot:  11
So chuoi lay duoc:  11
Series 1 length = 126
[5093. 5110. 5029. 5216. 

In [11]:
def split_train_test(series, h = 18):
  n = len(series)
  return series[:n-h], series[n-h:]

def make_supervised(arr, m):
  X, y = [], []
  for t in range(m, len(arr)):
    X.append(arr[t-m:t])
    y.append(arr[t])

  return np.array(X), np.array(y)

def SMAPE(y_true, y_pred):
  y_true = np.asarray(y_true, float)
  y_pred = np.asarray(y_pred, float)

  smape = []
  for i in range(len(y_true)):
    yt = y_true[i]
    yp = y_pred[i]
    numerator = abs(y_true[i] - y_pred[i])
    denominator = (abs(yt) + abs(yp)) / 2

    ratio = 0 if denominator == 0 else numerator / denominator
    smape.append(ratio)

  return 100*np.mean(smape)




In [12]:
def NARX(hidden, solver):
  return MLPRergessor(
      hidden_layer_sizes = (hidden,),
      solver = solver,
      activation = 'tanh',
      max_iter = 3000,
      random_state = 0
  )

In [13]:
def ten_fold_CV(trainval, m, hidden, solver, r, h, k):
  s = np.asarray(trainval, float)
  n = len(s)

  r_i = min(r, n - h - 1)
  if r_i < m or n - h <= r_i:
    return np.nan

  v = n - r- h
  if v <= 0:
    return np.nan
  d = v // k if v >= k else 1
  smape = []

  for i in range(k):
    end_train = r + i*d
    if end_train + h > n:
      break
    train = s[:end_train]

    #scale theo train
    scaler = MinMaxScaler(feature_range=(-1, 1))
    train_sc = scaler.fit_transform(train.reshape(-1, 1)).ravel()

    #Tao input autoregressive cho NARX
    Xtr, ytr = make_supervised(train_sc, m)
    if len(Xtr) == 0:
      continue

    #Train model tai fold i
    model = NARX(hidden, solver)
    model.fit(Xtr, ytr)

    #predict h buoc ke tiep
    hist = list(train_sc[-m:])
    preds_sc = []
    for _ in range(h):
      x_in = np.asarray(hist[-m:]).reshape(1, -1)
      yhat = model.predict(x_in)[0]
      preds_sc.append(yhat)
      hist.append(yhat)

    preds = scaler.inverse_transform(np.array(preds_sc).reshape(-1, 1)).ravel()
    y_true = s[end_train:end_train+h]
    smape.append(SMAPE(y_true, preds))

  return np.mean(smape) if smape else np.nan

In [14]:
def MCCV(trainval, m, hidden, solver, r, h, k):
  s = np.asarray(trainval, float)
  n = len(s)
  smape = []

  r_i = min(r, n - h - 1)
  if r_i < m or n - h <= r_i:
    return np.nan

  if n - h <= r:
    return np.nan

  for _ in range(k):
    end_train = random.randint(r, n-h)
    train = s[:end_train]

    scaler = MinMaxScaler(feature_range=(-1, 1))
    train_sc = scaler.fit_transform(train.reshape(-1, 1)).ravel()

    Xtr, ytr = make_supervised(train_sc, m)
    if len(Xtr) == 0:
      continue

    model = NARX(hidden, solver)
    model.fit(Xtr, ytr)

    hist = list(train_sc[-m:])
    preds_sc = []
    for _ in range(h):
      x_in = np.asarray(hist[-m:]).reshape(1, -1)
      yhat = model.predict(x_in)[0]
      preds_sc.append(yhat)
      hist.append(yhat)

    preds = scaler.inverse_transform(np.array(preds_sc).reshape(-1, 1)).ravel()
    y_true = s[end_train:end_train+h]
    smape.append(SMAPE(y_true, preds))

  return np.mean(smape) if smape else np.nan

In [15]:
def forecast_on_test(trainval, test, m, hidden, solver, h):
  s = np.asarray(trainval, float)
  t = np.asarray(test, float)

  scaler = MinMaxScaler(feature_range=(-1, 1))
  tr_sc = scaler.fit_transform(s.reshape(-1, 1)).ravel()

  Xtr, ytr = make_supervised(tr_sc, m)
  if len(Xtr) == 0:
    return np.nan

  model = NARX(hidden, solver)
  model.fit(Xtr, ytr)

  hist = list(tr_sc[-m:])
  preds_sc = []

  for _ in range(h):
    x_in = np.asarray(hist[-m:]).reshape(1, -1)
    yhat = model.predict(x_in)[0]
    preds_sc.append(yhat)
    hist.append(yhat)

  preds = scaler.inverse_transform(np.array(preds_sc).reshape(-1, 1)).ravel()
  return SMAPE(t, preds)

In [16]:
def main(filepath, r, h, k):
  print(" ~~ Doc file du lieu ~~")
  df = pd.read_excel(filepath, header = None)
  print(" ~~ Doc file du lieu thanh cong ~~")

  series_list = []
  for i in range(df.shape[0]):
    row = df.iloc[i, :]
    row = pd.to_numeric(row, errors = 'coerce').dropna().values.astype(float)
    series_list.append(row)

  print(f"So chuoi doc duoc: {len(series_list)}")

  param_grid = []
  for m in range(5, 11):
    for hidden in range(15, 21):
      for solver in ['lbfgs', 'sgd', 'adam']:
        param_grid.append({'m':m, 'hidden': hidden, 'solver': solver})

  s = len(series_list)
  M = len(param_grid)

  A = np.zeros((s, M))
  B = np.zeros((s, M))
  P = np.zeros((s, M))

  print(f"~~ Bat dau chay thu nghiem ~~")

  for i, serie in enumerate(series_list):
    print(f"~~ Chuoi thu {i+1}/{s} ~~")
    trainval, test = split_train_test(serie, h)
    for j, param in enumerate(param_grid):
      m = param['m']
      hidden = param['hidden']
      solver = param['solver']

      A[i, j] = ten_fold_CV(trainval, m, hidden, solver, r, h, k)
      B[i, j] = MCCV(trainval, m, hidden, solver, r, h, k)
      P[i, j] = forecast_on_test(trainval, test, m, hidden, solver, h)

  print(f"~~ Ket thuc chay thu nghiem ~~")
  return A, B, P, param_grid

In [17]:
A, B, P, param_grid = main(FILE, r=79, h=18, k=10)


 ~~ Doc file du lieu ~~
 ~~ Doc file du lieu thanh cong ~~
So chuoi doc duoc: 11
~~ Bat dau chay thu nghiem ~~
~~ Chuoi thu 1/11 ~~
~~ Chuoi thu 2/11 ~~


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS R

~~ Chuoi thu 3/11 ~~


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS R

~~ Chuoi thu 4/11 ~~
~~ Chuoi thu 5/11 ~~


...

~~ Chuoi thu 9/11 ~~


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS R

~~ Chuoi thu 10/11 ~~


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS R

~~ Chuoi thu 11/11 ~~
~~ Ket thuc chay thu nghiem ~~


In [18]:
def build_table_I(A, B, P):
    S, M = A.shape

    MSE_A_series = np.mean((P - A)**2, axis=1)
    MSE_B_series = np.mean((P - B)**2, axis=1)

    df = pd.DataFrame({
        "Series": np.arange(1, S+1),
        "MSE Ten-fold": MSE_A_series,
        "MSE MCCV": MSE_B_series
    })

    df.loc["Average"] = ["-", np.mean(MSE_A_series), np.mean(MSE_B_series)]
    df.loc["Std"] = ["-", np.std(MSE_A_series), np.std(MSE_B_series)]

    return df


In [19]:
def build_table_II(A, B, P):
    S, M = A.shape

    # MSE theo series
    MSE_A_series = np.mean((P - A)**2, axis=1)
    MSE_B_series = np.mean((P - B)**2, axis=1)

    # MSE theo models
    MSE_A_models = np.mean((P - A)**2, axis=0)
    MSE_B_models = np.mean((P - B)**2, axis=0)

    df = pd.DataFrame({
        "CV": ["Ten-fold", "MCCV"],
        "Average MSE": [
            np.mean((P - A)**2),
            np.mean((P - B)**2)
        ],
        "SD by series": [
            np.std(MSE_A_series),
            np.std(MSE_B_series)
        ],
        "SD by models": [
            np.std(MSE_A_models),
            np.std(MSE_B_models)
        ]
    })

    return df


In [20]:
table_I = build_table_I(A, B, P)
table_II = build_table_II(A, B, P)

print("=== TABLE I ===")
display(table_I.round(3))

print("=== TABLE II ===")
display(table_II.round(3))


=== TABLE I ===


,Series,MSE Ten-fold,MSE MCCV
0,1,30.134,37.691
1,2,235.526,325.342
2,3,665.630,659.088
3,4,234.274,232.951
4,5,260.780,259.701
5,6,151.449,163.239
6,7,56.645,59.614
7,8,42.146,38.369
8,9,938.812,954.384
9,10,856.398,909.497


=== TABLE II ===


,CV,Average MSE,SD by series,SD by models
0,Ten-fold,343.575,310.591,382.185
1,MCCV,356.951,318.185,397.520
